In [ ]:
# here about case in colab
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/CV_benthos_ws/'

In [ ]:
# in case of working in Colab or if there is no ultralytics on local machine
pip install ultralytics

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

In [ ]:
# Load YOLOv8 fine-tuned model 
#model = YOLO("yolov8n.pt")
model = YOLO("/content/drive/MyDrive/CV_benthos_ws/runs/detect/train/weights/best.pt")

#names for input and output file
video_path = path + 'test.mp4'
output_path = video_path[:-4] + '_output.mp4'

print()
print(video_path)
print(output_path)

In [ ]:
# if local
'''
import sys
path = sys.path[0]
print(path)
model = YOLO("runs/detect/train/weights/best.pt")
video_path = path + '/train.mp4'
output_path = path + '/train_output.mp4'
'''

In [ ]:
# Video input / output
video_path = video_path
cap = cv2.VideoCapture(video_path)
out = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    cap.get(cv2.CAP_PROP_FPS),
    (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
)

In [ ]:
# model was fine-tuned to detection of 5 classes 
CLASS_NAMES = ["Ophiura", "Gersemia", "Sea star", "Sea grass", "Crustaceans"]

# BGR
COLORS = {
    1: (0, 255, 0),     # green
    4: (255, 0, 0),     # blue
    0: (0, 0, 255),     # red
    2: (0, 255, 255),   # yellow
    3: (255, 0, 255)    # magenta
}

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # run YOLO tracking
    results = model.track(frame, persist=True)

    # extract detections
    bboxes = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes else []
    classes = results[0].boxes.cls.cpu().numpy().astype(int) if results[0].boxes else []
    ids = results[0].boxes.id.cpu().numpy().astype(int) if results[0].boxes.id is not None else [-1]*len(bboxes)

    for box, cls, track_id in zip(bboxes, classes, ids):
        x1, y1, x2, y2 = map(int, box)

        # pick color based on class
        color = COLORS.get(cls, (255, 255, 255))

        # draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        # prepare label text
        class_name = CLASS_NAMES[cls]
        if track_id != -1:
            label = f"{class_name}"# ID:{track_id}"
        else:
            label = class_name

        # draw text background
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(frame, (x1, y1 - th - 4), (x1 + tw, y1), color, -1)

        # draw label text
        cv2.putText(frame, label, (x1, y1 - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 2)

    #cv2.imshow("Tracking", frame)

    #if cv2.waitKey(1) & 0xFF == ord('q'):
    #    break

cap.release()
cv2.destroyAllWindows()